# ✈️ Airflow Integration Example

Schedule `mlprep` pipelines with Apache Airflow.

> **Note**: Airflow requires dedicated installation. This shows DAG definition.

## ⚙️ Configuration Parameters

Modify these paths for your environment:

In [ ]:
# ========================================
# CONFIGURE THESE PATHS FOR YOUR ENVIRONMENT
# ========================================
INPUT_DATA_PATH = '/path/to/data/input.csv'
PIPELINE_PATH = '/path/to/pipelines/pipeline.yaml'

print(f'Input data: {INPUT_DATA_PATH}')
print(f'Pipeline: {PIPELINE_PATH}')
print('\n⚠️ Update these paths before deploying to Airflow!')

In [ ]:
dag_code = f'''
from datetime import datetime, timedelta
from airflow import DAG
from airflow.operators.bash import BashOperator

default_args = {{
    'owner': 'mlprep_user',
    'depends_on_past': False,
    'email_on_failure': False,
    'email_on_retry': False,
    'retries': 1,
    'retry_delay': timedelta(minutes=5),
}}

with DAG(
    'mlprep_pipeline_demo',
    default_args=default_args,
    description='Run mlprep pipeline',
    schedule_interval=timedelta(days=1),
    start_date=datetime(2023, 1, 1),
    catchup=False,
    tags=['mlprep', 'etl'],
) as dag:

    check_data = BashOperator(
        task_id='check_input_data',
        bash_command='[ -f {INPUT_DATA_PATH} ] && echo "Data exists" || exit 1',
    )

    run_mlprep = BashOperator(
        task_id='run_mlprep_transform',
        bash_command='mlprep run {PIPELINE_PATH} --streaming --memory-limit 4GB',
    )

    check_data >> run_mlprep
'''

print('📄 Airflow DAG Code:')
print(dag_code)

In [ ]:
with open('mlprep_dag.py', 'w') as f:
    f.write(dag_code.strip())
print('✅ Saved mlprep_dag.py')

## 📋 Deployment Steps

1. Update `INPUT_DATA_PATH` and `PIPELINE_PATH` above
2. Copy `mlprep_dag.py` to Airflow `dags/` folder
3. Ensure `mlprep` is in PATH on worker nodes
4. Enable the DAG in Airflow UI

In [ ]:
print('🔄 Airflow + mlprep Integration')
print('='*40)
print('Tasks:')
print('  1. check_input_data: Verify input exists')
print('  2. run_mlprep_transform: Execute pipeline')
print('\nSchedule: Daily, retries once on failure')